# Qdrant

## 개요 및 특징

- Qdrant는 고성능 벡터 검색을 위한 오픈소스 벡터 데이터베이스이다. 딥러닝 모델에서 추출한 임베딩 벡터를 저장하고, 유사도 기반 검색을 빠르게 수행할 수 있도록 설계되었다. 
- 주요 특징

  - **고성능 검색**: HNSW 기반의 Approximate Nearest Neighbor(ANN) 알고리즘을 사용하여 빠른 벡터 검색 제공
  - **Payload Filtering**: 메타데이터 기반의 정교한 필터링 기능 제공 (예: 나이, 태그 등 조건 지정 가능)
  - **분산 지원**: 샤딩, 레플리카 등의 분산 기능으로 확장성 확보
  - **Python, Javascript, Rust, Java 등 다양한 언어지원**: 시스템 통합, 개발 편의성, 다양한 애플리케이션 구축이 가능.
  - **REST API 및 gRPC 지원**: 다양한 클라이언트 환경에 대응 가능
  - **스냅샷 및 영속성 보장**: 내장된 스냅샷과 WAL 기능으로 데이터 백업과 복구를 지원.
  - **오픈소스 프로젝트**: **Apache 2.0 라이선스** 하에 배포되는 오픈소스 프로젝트로 기업 환경에서도 제약 없이 자유롭게 사용할 수 있으며, 수정 및 배포도 허용된다.
- 사이트
  - 홈페이지: https://qdrant.tech
  - GitHub: https://github.com/qdrant/qdrant

## Qdrant vs 다른 벡터 DB 비교

| 항목            | Qdrant               | FAISS           | Milvus             | Weaviate  |
| ------------- | -------------------- | --------------- | ------------------ | --------- |
| 검색 방식         | HNSW, Flat           | Flat, IVF, HNSW | IVF, HNSW, DiskANN | HNSW      |
| 필터링 지원        | ✅ (Boolean + Nested) | ❌               | ✅                  | ✅         |
| REST API      | ✅                    | ❌               | ✅                  | ✅         |
| gRPC 지원       | ✅                    | ❌               | ✅                  | ❌         |
| 벡터 + 메타데이터 통합 | ✅                    | ❌               | ✅                  | ✅         |
| 클라우드 서비스      | ✅ (Qdrant Cloud)     | ❌               | ✅                  | ✅         |
| 커뮤니티 및 문서     | 활발                   | 활발              | 활발                 | 활발        |
| 설치 용이성        | Docker로 간단           | 로컬 빌드 필요        | Docker 가능          | Docker 가능 |


# Qdrant 아키텍처

## Core Engine 구조

Qdrant의 Core Engine은 다음과 같은 구성 요소로 이루어집니다:

- **Search Engine**
  - HNSW 기반의 벡터 유사도 탐색 수행
- **Segment**
  - Segment는 QDrant 내부의 핵심 저장 및 검색 단위이다.
  - 데이터는 여러 Segment로 나뉘어 저장되며, 하나의 Collection은 여러 Segment로 구성된다.
  - 각 Segment는 **독립적인 인덱스와 저장 공간**을 가진다. 이를 통해 **검색의 정확도, 속도, 저장 효율성**을 높인다.
- **Index Manager**
  - 인덱스 상태, 설정, 생성 및 최적화를 담당
- **Write-Ahead Log (WAL)**
  - 데이터 변경작업(삽입, 수정, 삭제)가 발생하면 **디스크에 먼저 기록하고 이후 실제 메모리에 적용하는 방식**
  - 장애 발생시 WAL에 기록된 작업들을 재적용해서 데이터 복원이 가능

## Payload System (Meta-Data Storage)
- Qdrant는 각 벡터와 함께 JSON 형태의 payload를 저장할 수 있다. 
    - payload는 meta-data를 말한다.
- payload는 다음과 같은 다양한 타입을 지원:
    - string, integer, float, boolean, list
- payload는 **검색 시 필터링 조건**으로 활용 가능

## Filtering Mechanism
- Query 시 payload 조건을 지정하여 검색 범위를 제한 가능하여 검색 정확도를 높일 수있다.
- 다양한 검색 조건 비교를 지원하고 **nested 필터**를 통해 Payload의 중첩된 JSON 구조 내부의 필드에 대한 조건에 대해서도 필터링 가능하다.

## WAL(Write-Ahead Log) 및 Durability

- 모든 변경 사항(삽입, 삭제 등)은 WAL에 먼저 기록됨
- WAL은 디스크 기반이며 장애 발생 시 복구에 사용
- WAL + Snapshot을 함께 사용해 데이터 무결성과 내구성 확보할 수있다.s


# Qdrant 핵심 구성 요소

## Collection

- Collection은 Qdrant에서 벡터와 payload를 묶어 저장하는 **논리적 데이터베이스 단위** 이다. 
- SQL의 테이블과 유사한 개념으로, 하나의 Collection에는 동일한 차원의 벡터들이 저장된다.

## 주요 설정 항목
- `name`: 컬렉션 이름
- `vectors.size`: 임베딩 벡터 차원 수 (예: 768, 384 등)
- `vectors.distance`: 유사도 계산 방식 (Cosine, Dot, Euclidean)

## Point
- Point는 Qdrant에서 하나의 벡터 데이터 항목을 의미한다. 하나의 Point는 다음 세 가지로 구성된다.
    1. `id`: 고유 식별자 (정수 또는 UUID)
    2. `vector`: 임베딩된 고차원 벡터 값 (예: [0.12, 0.23, ..., 0.91])
    3. `payload`: 해당 벡터와 연결된 메타데이터 (JSON format)

### 사용 예시 (Python SDK)

  ```python
  client.upsert(
    collection_name="my_collection",
    points=[
      PointStruct(
        id=1,
        vector=[0.1, 0.2, 0.3],   # vector 크기는 collection 생성 시 지정한 vector size 와 동일해야 한다.
        payload={"category": "book", "price":20000}
      )
    ]
  )
  ```

### Payload
- Payload는 Point에 연결된 **메타데이터 정보**로, 검색 필터링시 사용된다. JSON 형식으로 저장되며, 다양한 데이터타입을 지원한다.
- Payload의 개별 원소는 **Field** 라고 하며 **key-value** 쌍으로 구성된다.
- 지원 데이터 타입
    - `string`, `number`, `boolean`, `array`, `nested object`

#### 예시

```json
{
  "category": "book",
  "rating": 4.5,
  "tags": ["fiction", "bestseller"],
  "user": {
    "name": "Alice",
    "age": 30
  }
}
```

#### 필터링에서의 활용

```python
condition_category = FieldCondition(
    key="category",
    match=MatchValue(value="book")
)

condition_user_age = FieldCondition(
    key="user.age",     # nested object
    range=Range(gte=25)
)

```

# Qdrant 설치 및 실행

## Python
- `pip install qdrant-client`
- **Local 컴퓨터의 메모리, HDD/SSD**를 이용해 데이터를 관리할 경우 추가 설치 없이 python lib를 이용해 연결할 수있다.

    ```python
    from qdrant_client import QdrantClient
    qdrant = QdrantClient(":memory:") # Create in-memory Qdrant instance, for testing, CI/CD
    # OR
    client = QdrantClient(path="path/to/db")  # Persists changes to disk, fast prototyping
    ```

## Server-Client

### Docker로 QDrant 서버 실행
- 이미지 다운로드 및 실행
    - 6333: REST API 포트
    - 6334: gRPC 포트

    ```bash
    docker pull qdrant/qdrant
    docker run -p 6333:6333 -p 6334:6334 qdrant/qdrant
    docker run -p 6333:6333 -p 6334:6334 -v "qdrant_storage:/qdrant/storage:z"  qdrant/qdrant
    ```
- 파이썬에서 연결

    ```python
    from qdrant_client import QdrantClient
    client = QdrantClient(url="http://localhost:6333")
    ```
## Qdrant Cloud
- https://cloud.qdrant.io
- 무료 tier 제공 (1GB 용량)
- 웹 UI로 Collection 생성 및 벡터 관리 가능
- 연결

    ```python
    client = QdrantClient(
        url=url,
        api_key=api_key,
    )
    ```


# 실습

## 연결, collection 생성

In [1]:
!uv pip install qdrant-client

Resolved 21 packages in 590ms
 Downloaded grpcio
Prepared 6 packages in 357ms
Installed 6 packages in 253ms
 + grpcio==1.81.1
 + h2==4.3.0
 + hpack==4.2.0
 + hyperframe==6.1.0
 + portalocker==3.2.0
 + qdrant-client==1.18.0


In [8]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance
###########################################
# 1. QDrant 연결
#    - 메모리
#    - 파일
#    - 서버
# 2. Collection 조회 및 생성   
###########################################
COLLECTION_NAME = "test_collection2"

# client = QdrantClient(":memory:") # 메모리에 연결
# client = QdrantClient(path="vector_storage/qdrant") #  Local File에 저장.

## QDrant server를 docker로 실행(터미널에서 실행) -> 연결
# docker run -p 6333:6333 -p 6334:6334 -v "qdrant_storage:/qdrant/storage:z" qdrant/qdrant
## qdrant dashboard에 접속 - http://localhost:6333/dashboard (웹브라우저)
##  - GUI환경으로 DB상태를 확인.
client = QdrantClient(url="http://localhost:6333")

# collection 존재여부 확인
exist = client.collection_exists(collection_name=COLLECTION_NAME)  # 이 이름의 컬랙션이 있는지 여부(bool)
print(exist)

# 만약에 있으면 삭제
if exist:
    client.delete_collection(collection_name=COLLECTION_NAME)

# collection 생성
client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(  # Dense Vector(embedding vector) 저장소에 대한 설정
        size=4, #embedding vector의 차원수. 사용할 임베딩 모델의 차원수에 맞춰준다. (text-embedding-3-large: 3072)
        distance=Distance.COSINE, # 유사도 계산 방법 - COSINE:코사인유사도, EUCLID(유클리디안), DOT(내적)
    )
)

print(client.get_collections()) # QDrant DB의 모든 컬렉션들을 조회

client.close()

False
collections=[CollectionDescription(name='test_collection2')]


## 데이터 삽입
- `upsert()`

In [9]:
from qdrant_client.models import  PointStruct
# Point - 개별데이터: PointStruct - id, vector, payload
dataset = [
        PointStruct(id=1, vector=[0.05, 0.61, 0.76, 0.74], payload={"city": "Berlin", "country":"Germany", "num":1000}),
        PointStruct(id=2, vector=[0.19, 0.81, 0.75, 0.11], payload={"city": "London", "country":"England", "num":5000}),
        PointStruct(id=3, vector=[0.36, 0.55, 0.47, 0.94], payload={"city": "Moscow", "country":"Russia", "num":6000}),
        PointStruct(id=4, vector=[0.18, 0.01, 0.85, 0.80], payload={"city": "New York", "country":"USA", "num":10000}),
        PointStruct(id=5, vector=[0.28, 0.03, 0.15, 0.10], payload={"city": "LA", "country":"USA", "num":20000}),
        PointStruct(id=6, vector=[0.24, 0.18, 0.22, 0.44], payload={"city": "Beijing", "country":"China", "num":15000}),
        PointStruct(id=7, vector=[0.35, 0.08, 0.11, 0.44], payload={"city": "Mumbai", "country":"India", "num":16000}),
        PointStruct(id=8, vector=[0.49, 0.77, 0.75, 0.31], payload={"city": "Liverpool", "country":"England", "num":17000}),
        PointStruct(id=9, vector=[0.43, 0.72, 0.12, 0.38], payload={"city": "Shanghai", "country":"China", "num":123000}),
]


In [10]:
# Qdrant DB와 연결.
client = QdrantClient(url="http://localhost:6333")
upsert_info = client.upsert(
    collection_name=COLLECTION_NAME, # 데이터를 저장할 collection 이름
    points=dataset,   # 저장할 데이터셋. list[PointStruct]
    wait=True         # WAL (저장하는 것을 로그로 남기고 저장.) True: 완전히 저장 후 응답. False: 로그만 작성하고 응답
)

In [11]:
upsert_info

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [ ]:
client.close()

## 검색

### 유사도 검색
- `query_points()`

In [13]:
# 연결
client = QdrantClient(url="http://localhost:6333")
# 검색 - 유사도(의미-semantic search) 기반 검색
search_result = client.query_points(
    collection_name= COLLECTION_NAME,
    query=[0.2, 0.1, 0.9, 1.2], # 질문의 embedding vector 
    limit=3, # top-k (몇개를 조회할 지.)
)

In [22]:
# search_result
for point in search_result.points:
    print("유사도점수:", point.score, ", 문서 id:", point.id, ", 페이로드:", point.payload)
    print("------------------------------------------")

유사도점수: 0.98368233 , 문서 id: 4 , 페이로드: {'city': 'New York', 'country': 'USA', 'num': 10000}
------------------------------------------
유사도점수: 0.90634227 , 문서 id: 6 , 페이로드: {'city': 'Beijing', 'country': 'China', 'num': 15000}
------------------------------------------
유사도점수: 0.892581 , 문서 id: 3 , 페이로드: {'city': 'Moscow', 'country': 'Russia', 'num': 6000}
------------------------------------------


In [23]:
client.close()

## Filtering

- Qdrant의 Filtering은 **벡터 검색과 함께 payload(메타데이터)에 대해 조건을 먼저 적용**하는 기능이다.
- Filtering 적용시 검색 흐름(순서)

  1. Payload 조건으로 후보 벡터를 먼저 필터링
  2. 필터링된 벡터에 대해서만 유사도 검색 수행

- Filtering의 효과.

  - 검색 정확도 향상 (불필요한 벡터 제거)
  - 검색 속도 향상 (탐색 공간 축소)
  - 현실적인 검색 조건(가격, 지역, 카테고리, 상태 등)과 벡터 검색을 결합하여 검색할 수 있다.
  - RAG, 추천 시스템, 하이브리드 검색의 필수 구성 요소

- 다양한 예제: - https://qdrant.tech/documentation/concepts/filtering/

### Filter의 전체 구조 (Filter Clauses)

- 여러 조건을 논리 연산으로 결합하는 구조이다.
- 논리 연산자 역할
  
| 절            | 의미                     | 논리 연산 대응   |
| ------------ | ---------------------- | ---------- |
| `must`       | 모든 조건을 반드시 만족         | AND        |
| `should`     | 조건 중 하나 이상 만족          | OR         |
| `min_should` | `should` 중 최소 k개 이상 만족 | OR + 최소 개수 |
| `must_not`   | 해당 조건을 절대 만족하면 안 됨     | NOT        |

- **이 네 가지 구조는 자유롭게 조합 및 중첩 가능하다.**
- 예시

  ```python
  query_filter = Filter(
      must=[
          # category == "book"
          FieldCondition(
              key="category",
              match=MatchValue(value="book")
          ),

          # title match_text "러닝"
          FieldCondition(
              key="title",
              match=MatchText(text="러닝")
          ),
      ],
      must_not=[
          # is_soldout == true 인 데이터 제외
          FieldCondition(
              key="is_soldout",
              match=MatchValue(value=True)
          )
      ]
  )
  ```
- “카테고리는 book이고 title에 러닝이 들어가면서, 품절되지 않은(is_soldout 이 true가 아닌) 상품만 검색”


### 조건 타입 (Condition Types)

- Filtering에서 실제로 사용하는 **payload 조건 연산자들**이다.

- 구문패턴
  - `FieldCondition(key="payload_key", 조건_연산자=Value객체)`
  
- **조건 연산자들**
  -  `match` 계열 (값 비교 조건)
    -  정확 일치 (`=`)

        ```python
        condition = FieldCondition(
            key="brand",
            match=MatchValue(value="Nike")
        )

        ```

  - `match_any`
    -  여러 값 중 하나 일치 (IN)

          ```python
          condition = FieldCondition(
              key="color",
              match_any=MatchAny(values=["red", "blue"])
          )
          ```
  - `match_except` 
    - 특정 값들 제외 (NOT IN)

      ```python
      condition = FieldCondition(
          key="status",
          match_except=MatchExcept(values=["deleted"])
      )
      ```
  - `match_text`
    - 텍스트 검색. 지정한 payload 값을 단어(토큰) 단위로 분해한 뒤, 해당 단어가 포함된 문서를 검색한다.
    - SQL의 like와 같이 부분 문자열 포함을 검색하지만 like와 같이 와일드 카드를 이용해 자유롭게 검색하는 것은 아니다.
  
      ```python
        condition = FieldCondition(
            key="description",
            match=MatchText(text="스니커즈")
        )
        ```
  - `range` 
    - 수치형, 날짜형 데이터에 대해 **범위 조건**을 설정한다.
    - 연산자
      | 연산자   | 의미 |
      | ----- | -- |
      | `gt`  | 초과 |
      | `gte` | 이상 |
      | `lt`  | 미만 |
      | `lte` | 이하 |

        ```python
        condition = FieldCondition(
            key="price",
            range=Range(gte=10000, lt=20000)
        )
        ```

  -  `exists`    
     - key가 존재 하는 지 여부
  
        ```python
        condition = HasFieldCondition(key="price")
        ``` 
     - price key가 있는지 여부

  - `is_empty`
    - key에 값이 비어있는가?
    - key는 있는데 거기에 **값이 비어있는지 여부**
      ```python
        condition = IsEmptyCondition(key="reviews")
      ```
    -  reviews key는 존재하지만 값이 [] 이거나 'null' 인 데이터를 검색. (빈문자열은 false-버전에 따라 다름.)

  - `values_count`
    - 배열형 payload의 **값의 개수** 조건을 지정.

      ```python
          condition = FieldCondition(
            key="tags",
            values_count=ValuesCount(gte=2)
          )
      ```
    -  태그가 2개 이상인 데이터만 검색
    
  -  `geo`
     -  위경도 좌표 payload에 대해 사용하는 **거리 기반 필터링**이다.

          ```python
            condition = FieldCondition(
                key="location",
                geo_radius=GeoRadius(
                    center=GeoPoint(lat=37.5, lon=127.0),
                    radius=3000
                )
            )
          ```

### Filtering과 Payload Index
- Payload Index 없는 field에 대해 Filtering을 하면 기본적으로 모든 데이터에 대해 조건 검색을 하게 된다(Full Scan). 거기에 유사도 검색까지 하게 되면 검색이 너무 오래 걸리게 된다.
- 이 문제를 해결하기 위해 Payload에 index를 걸어 줄 수 있다. Payload index는 필드 타입에 따라 B-Tree, 해시 인덱스, inverted index 등 다양한 내부 인덱스 구조를 사용하여 빠르게 검색할 수 있도록 해준다.
- 다음 필드들에 index를 만들어 주는 것이 좋다.
  - 자주 filtering 하는 숫자 필드 (price, age, score)
  - 범주 값을 저장하는 필드 (category, type, status)
  - 날짜 (created_at, updated_at)
  - 위치 (geo index)
  - boolean 값을 가지는 필드
- payload 설정
  ```python
  client.create_payload_index(
      collection_name="products",  # collection 이름
      field_name="price",          # index를 걸어줄 payload의 field name
      field_schema=PayloadSchemaType.INTEGER  # field의 데이터 타입. 타입에 맞는 index 알고리즘을 사용한다.
  )
  ```
- field type
  - `PayloadSchemaType.KEYWORD`: 문자열. 일치 검색일 때 지정
  - `PayloadSchemaType.TEXT`: 문자열. match_text (특정 단어를 포함하고 있는지를 검색일 때 지정)
  - `PayloadSchemaType.INTEGER`
  - `PayloadSchemaType.FLOAT`
  - `PayloadSchemaType.GEO`
  - `PayloadSchemaType.BOOL`
  - `PayloadSchemaType.DATETIME`
  - `PayloadSchemaType.UUID`
  - 문자열 타입의 필드
    - `match`(==), `match_any`, `match_except` 비교를 주로 할 경우 `PayloadSchemaType.KEYWORD` 로 설정한다.
      - 범주형 field에 설정. (카테고리, 상태값, 타입, 태그등)
    - `match_text` 비교를 주로 할 경우 `PayloadSchemaType.TEXT`로 설정한다.
      - 긴 문서 field에 설정한다. (설명, 리뷰내용 등)

In [1]:
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Filter, # Filter 식을 만들때 사용
    FieldCondition, # 개별 조건 만들때 사용
    MatchValue, #  == 일치 검색연산
    MatchText, # 부분일치 검색 연산
    Range,     # 범위(>, <, >=, <=) 검색 연산
    PayloadSchemaType # Payload Field에 Index의 타입을 제공.
) 
COLLECTION_NAME = "test_collection2"

In [ ]:
# docker run -p 6333:6333 -p 6334:6334 -v "qdrant_storage:/qdrant/storage:z" qdrant/qdrant

In [2]:
client = QdrantClient(url="http://localhost:6333")

# Index 설정.
client.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="country", # Index를 설정할 payload key(Field)
    field_schema=PayloadSchemaType.KEYWORD
)
client.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="city", # Index를 설정할 payload key(Field)
    field_schema=PayloadSchemaType.KEYWORD
)
client.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="num", # Index를 설정할 payload key(Field)
    field_schema=PayloadSchemaType.INTEGER
)

UpdateResult(operation_id=13, status=<UpdateStatus.COMPLETED: 'completed'>)

In [3]:
collection_info = client.get_collection(COLLECTION_NAME) #  지정한 collection 정보를 반환.
# collection의 index 정보를 조회
collection_info.payload_schema

{'num': PayloadIndexInfo(data_type=<PayloadSchemaType.INTEGER: 'integer'>, params=None, points=9),
 'city': PayloadIndexInfo(data_type=<PayloadSchemaType.KEYWORD: 'keyword'>, params=None, points=9),
 'country': PayloadIndexInfo(data_type=<PayloadSchemaType.KEYWORD: 'keyword'>, params=None, points=9)}

In [4]:
#######################
# Payload Filtering 조회
#
# Vector 유사도 검색 + Payload Filtering
##########################
search_result = client.query_points(
    collection_name=COLLECTION_NAME,
    query=[0.2, 0.3, 0.9, 1.2], 
    limit=3,
    query_filter=Filter(
        must=[
            FieldCondition(
                key="country", # 검색할 Field
                match=MatchValue(value="USA") # payload[country] == 'USA'
            ),
            # FieldCondition(
            #     key="city", # 검색할 Field
            #     match=MatchValue(value="Seatle") # payload[country] == 'USA'
            # ),
        ]
    )
)
for point in search_result.points:
    print(point)

id=4 version=1 score=0.96810615 payload={'city': 'New York', 'country': 'USA', 'num': 10000} vector=None shard_key=None order_value=None
id=5 version=1 score=0.6203555 payload={'city': 'LA', 'country': 'USA', 'num': 20000} vector=None shard_key=None order_value=None


In [36]:
search_result = client.query_points(
    collection_name=COLLECTION_NAME,
    query=[0.2, 0.3, 0.9, 1.2], 
    limit=3,
    query_filter=Filter(
        must=[
            FieldCondition(
                key="num", # 검색할 Field
                # gt, gte, lt, lte
                range=Range(gt=10000, lt=20000) # 범위 조회 payload[num] > 10000  and payload[num] < 20000
            )
        ]
    )
)
for point in search_result.points:
    print(point)

id=6 version=1 score=0.9314785 payload={'city': 'Beijing', 'country': 'China', 'num': 15000} vector=None shard_key=None order_value=None
id=7 version=1 score=0.8079488 payload={'city': 'Mumbai', 'country': 'India', 'num': 16000} vector=None shard_key=None order_value=None
id=8 version=1 score=0.7303041 payload={'city': 'Liverpool', 'country': 'England', 'num': 17000} vector=None shard_key=None order_value=None


In [5]:
search_result = client.query_points(
    collection_name=COLLECTION_NAME,
    query=[0.2, 0.3, 0.9, 1.2], 
    limit=3,
    query_filter=Filter(
        should=[
            FieldCondition(
                key="num", # 검색할 Field
                range=Range(gt=10000, lt=20000) # 범위 조회 payload[num] > 10000  and payload[num] < 20000
            ),
            FieldCondition(
                key="city",
                match=MatchText(text="Be")
            )
        ]
    )
)
for point in search_result.points:
    print(point)

id=1 version=1 score=0.93419933 payload={'city': 'Berlin', 'country': 'Germany', 'num': 1000} vector=None shard_key=None order_value=None
id=6 version=1 score=0.9314785 payload={'city': 'Beijing', 'country': 'China', 'num': 15000} vector=None shard_key=None order_value=None
id=7 version=1 score=0.8079488 payload={'city': 'Mumbai', 'country': 'India', 'num': 16000} vector=None shard_key=None order_value=None


In [6]:
from qdrant_client.models import MinShould

search_result = client.query_points(
    collection_name=COLLECTION_NAME,
    query=[0.2, 0.3, 0.9, 1.2],
    limit=3,
    query_filter=Filter(
        min_should=MinShould(
            min_count=2, # 조건들 중 최소 2개는 True인 Data Point 조회
            conditions=[
                FieldCondition(key="country", match=MatchValue(value="USA")),
                FieldCondition(key="city", match=MatchText(text="Be")),
                FieldCondition(key="num", range=Range(lt=5000)),
            ]
        )
    )
)

In [7]:
for point in search_result.points:
    print(point)

id=1 version=1 score=0.93419933 payload={'city': 'Berlin', 'country': 'Germany', 'num': 1000} vector=None shard_key=None order_value=None


In [13]:
type(search_result)

qdrant_client.http.models.models.QueryResponse

In [8]:
client.close()

## 단순 조회 
- `scroll()`

In [19]:
client = QdrantClient(url="http://localhost:6333")

# 반환: (조회결과 dataset:list[Record], 다음에 가져올 시작 Field의 id-next offset)
result, next_field_id = client.scroll(
    collection_name=COLLECTION_NAME,
    limit=3, # 가져올 개수
)
print("다음에 가져올 필드 ID:", next_field_id)
for r in result:
    print(r.id, r.payload)

다음에 가져올 필드 ID: 4
1 {'city': 'Berlin', 'country': 'Germany', 'num': 1000}
2 {'city': 'London', 'country': 'England', 'num': 5000}
3 {'city': 'Moscow', 'country': 'Russia', 'num': 6000}


In [20]:
result, next_field_id = client.scroll(
    collection_name=COLLECTION_NAME,
    limit=6, # 가져올 개수
    offset=next_field_id, # 데이터를 가져올 시작 Field의 ID. 생략: 처음부터
)
print("다음에 가져올 필드 ID:", next_field_id)
for r in result:
    print(r.id, r.payload)

다음에 가져올 필드 ID: None
4 {'city': 'New York', 'country': 'USA', 'num': 10000}
5 {'city': 'LA', 'country': 'USA', 'num': 20000}
6 {'city': 'Beijing', 'country': 'China', 'num': 15000}
7 {'city': 'Mumbai', 'country': 'India', 'num': 16000}
8 {'city': 'Liverpool', 'country': 'England', 'num': 17000}
9 {'city': 'Shanghai', 'country': 'China', 'num': 123000}


## 삭제
- `delete()`

In [21]:
client.delete(
    collection_name=COLLECTION_NAME,
    points_selector=[1, 3] # 삭제할 포인트의 ID
)

UpdateResult(operation_id=14, status=<UpdateStatus.COMPLETED: 'completed'>)

In [28]:
# Field Filtering 이용해서 삭제
client.delete(
    collection_name=COLLECTION_NAME,
    points_selector=Filter(  # Filter객체 -> field filtering 조건으로 삭제
        must=[FieldCondition(key='country', match=MatchValue(value='England'))]
    )
)

UpdateResult(operation_id=15, status=<UpdateStatus.COMPLETED: 'completed'>)

In [29]:
result = client.scroll(
    collection_name=COLLECTION_NAME,
    limit=10
)
result

([Record(id=4, payload={'city': 'New York', 'country': 'USA', 'num': 10000}, vector=None, shard_key=None, order_value=None),
  Record(id=5, payload={'city': 'LA', 'country': 'USA', 'num': 20000}, vector=None, shard_key=None, order_value=None),
  Record(id=6, payload={'city': 'Beijing', 'country': 'China', 'num': 15000}, vector=None, shard_key=None, order_value=None),
  Record(id=7, payload={'city': 'Mumbai', 'country': 'India', 'num': 16000}, vector=None, shard_key=None, order_value=None),
  Record(id=9, payload={'city': 'Shanghai', 'country': 'China', 'num': 123000}, vector=None, shard_key=None, order_value=None)],
 None)

In [31]:
# client.delete(
#     collection_name=COLLECTION_NAME,
# )

## 수정

In [35]:
# 데이터 Point 수정 - upsert(): 있는 ID면 수정, 없는 ID면 추가
from qdrant_client.models import PointStruct
p = PointStruct(
    id=10,
    vector=[1.2, 0.3, -0.07, 2.4],
    payload={"city":"Seoul", "country":"Korea", "num":500000}
)
client.upsert(
    collection_name=COLLECTION_NAME,
    points=[p]
)


UpdateResult(operation_id=17, status=<UpdateStatus.COMPLETED: 'completed'>)

In [37]:
#################
# payload 를 수정
#################
# 특정 Field의 값을 변경 - set_payload()
client.set_payload(
    collection_name=COLLECTION_NAME,
    payload={"city":"Shanghai", "info":"중국 대표 도시"}, # 변경(city), 추가(info) 할 것을 dict에 설정.
    points=[6], # 수정할 DataPoint 의 id
)

UpdateResult(operation_id=18, status=<UpdateStatus.COMPLETED: 'completed'>)

In [40]:
# payload 자체를 변경
client.overwrite_payload(
    collection_name=COLLECTION_NAME,
    payload={"도시이름":"인천", "나라":"대한민국", "정보":"광역시"},
    points=[5]
)

UpdateResult(operation_id=19, status=<UpdateStatus.COMPLETED: 'completed'>)

In [42]:
# payload의 특정 Field를 삭제

client.delete_payload(
    collection_name=COLLECTION_NAME,
    keys=["city", "num"], # 삭제할 key들을 지정
    points=[4, 6], # 삭제할 Point의 id
)

UpdateResult(operation_id=20, status=<UpdateStatus.COMPLETED: 'completed'>)

In [44]:
# Payload 자체를 제거/삭제
client.clear_payload(
    collection_name=COLLECTION_NAME,
    points_selector=[7, 9]
)

UpdateResult(operation_id=21, status=<UpdateStatus.COMPLETED: 'completed'>)

In [45]:
result, _ = client.scroll(
    collection_name=COLLECTION_NAME, limit=10
)
result

[Record(id=4, payload={'country': 'USA'}, vector=None, shard_key=None, order_value=None),
 Record(id=5, payload={'도시이름': '인천', '나라': '대한민국', '정보': '광역시'}, vector=None, shard_key=None, order_value=None),
 Record(id=6, payload={'info': '중국 대표 도시', 'country': 'China'}, vector=None, shard_key=None, order_value=None),
 Record(id=7, payload={}, vector=None, shard_key=None, order_value=None),
 Record(id=9, payload={}, vector=None, shard_key=None, order_value=None),
 Record(id=10, payload={'city': 'Seoul', 'country': 'Korea', 'num': 500000}, vector=None, shard_key=None, order_value=None)]